In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import torch
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from models.lstm_model import LSTM_Model
from utils import EarlyStopping, get_device, EpochTrainer, build_seq, get_yf_data
from operator import itemgetter
from sklearn.preprocessing import RobustScaler as Scaler

# Parameters

In [3]:
model_output_filename = "lstm_checkpoint"

## model config

In [4]:
batch_size = 100
epochs = 100
hidden_size = 64
num_output = 1
num_layers = 2
dropout = 0.1
bidirectional = False
patience = 10
test_size = 0.3
lr = 0.001

In [5]:
seq_len = 30
data_len = 500


eval_method = 'RMSE'
criterion = nn.MSELoss()

In [6]:
input_cols = ["Open", "High", "Low", "Close", "Volume"]
identifier_cols = ['Date']
target_cols = ['Close']

# Get Sample Data

## Dataset additional Parameters

In [7]:
steps_ahead = 5
ticker = "VOO"
start = "2015-01-01"
end = "2026-04-29"

## Load data from yfinance

In [8]:
data = get_yf_data( ticker , start , end)

[*********************100%***********************]  1 of 1 completed


# Data Processing

## Scale inputs and target

In [9]:
x_scaler = Scaler()
y_scaler = Scaler()

X_all = x_scaler.fit_transform(data[input_cols].values)
y_all = y_scaler.fit_transform(data[target_cols].values)

## Build Sequence

In [10]:
id_all = data[identifier_cols].values.flatten().tolist()

Seqs = build_seq( seq_len , X_all, steps_ahead, id_all , y_all  )

X_seq, y_seq, X_id, y_id = itemgetter('X_Seq', 'y_Seq', 'X_id', 'y_id')(Seqs)

In [11]:
X_seq, y_seq, X_id, y_id = X_seq[-data_len:], y_seq[-data_len:], X_id[-data_len:], y_id[-data_len:]

## Convert to Tensor

In [12]:
X_tensor = torch.from_numpy(np.asarray(X_seq, dtype=np.float32))
y_tensor = torch.tensor(np.array(y_seq), dtype=torch.float32)

In [13]:
full_dataset = TensorDataset(X_tensor, y_tensor)

## Split data into training and testing

In [14]:
input_size = X_tensor.shape[-1]

In [15]:
test_records = int(test_size * len(full_dataset))
train_records = len(full_dataset) - test_records

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_records, test_records],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Prepare Model

## Prepare config of the model

In [16]:
preprocess_config = {
    "seq_length": seq_len,
    "input_size": input_size,
    "task_type": "classification",
}

In [17]:
model_config = {
    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_layers": num_layers,
    "dropout": dropout,
    "bidirectional": bidirectional,
}

## Get Model

In [18]:
device = get_device()

In [19]:
model = LSTM_Model(
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    num_layers=num_layers,
    dropout=dropout,
    bidirectional=bidirectional,
).to(device)

In [20]:
optimizer = torch.optim.Adam(model.parameters(), lr = lr)

## Early stopping and checkpoint setup

In [21]:
early_stopping = EarlyStopping(
    patience=patience,
    path=f"../checkpoints/{model_output_filename}.pt",
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "target_cols": target_cols,
        "input_cols": input_cols,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "steps_ahead" : steps_ahead,
    },
)

# Loop epochs and train model

In [22]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = eval_method
)

In [23]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        
        print(f"Epoch {epoch + 1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch   1 | Train Loss: 1.9844 | Val Loss: 1.7908 | RMSE: 1.3382
Epoch   2 | Train Loss: 1.5323 | Val Loss: 1.3042 | RMSE: 1.1420
Epoch   3 | Train Loss: 1.0109 | Val Loss: 0.5952 | RMSE: 0.7715
Epoch   4 | Train Loss: 0.2907 | Val Loss: 0.0380 | RMSE: 0.1949
Epoch   5 | Train Loss: 0.1793 | Val Loss: 0.2082 | RMSE: 0.4563
Epoch   6 | Train Loss: 0.1454 | Val Loss: 0.0369 | RMSE: 0.1922
Epoch   7 | Train Loss: 0.0548 | Val Loss: 0.0799 | RMSE: 0.2826
Epoch   8 | Train Loss: 0.0880 | Val Loss: 0.0954 | RMSE: 0.3088
Epoch   9 | Train Loss: 0.0783 | Val Loss: 0.0582 | RMSE: 0.2412
Epoch  10 | Train Loss: 0.0503 | Val Loss: 0.0373 | RMSE: 0.1931
Epoch  11 | Train Loss: 0.0477 | Val Loss: 0.0413 | RMSE: 0.2031
Epoch  12 | Train Loss: 0.0496 | Val Loss: 0.0382 | RMSE: 0.1955
Epoch  13 | Train Loss: 0.0462 | Val Loss: 0.0352 | RMSE: 0.1876
Epoch  14 | Train Loss: 0.0412 | Val Loss: 0.0384 | RMSE: 0.1960
Epoch  15 | Train Loss: 0.0425 | Val Loss: 0.0378 | RMSE: 0.1944
Epoch  16 | Train Loss: 0